<a href="https://colab.research.google.com/github/rishan005/AI-Architecture/blob/main/AI_Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q "diffusers>=0.27" transformers accelerate safetensors
!pip install -q py360convert Pillow numpy scipy
!pip install -q gradio huggingface_hub
print(" All packages installed!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/360RoomEditor_v6'
for sub in ['uploads', 'outputs', 'references', 'masks']:
    os.makedirs(f'{PROJECT_DIR}/{sub}', exist_ok=True)
print(f" Drive mounted → {PROJECT_DIR}")

In [ ]:
import gc, shutil, traceback, base64, io, math
from datetime import datetime

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageFilter, ImageDraw, ImageOps, ImageEnhance
import py360convert
from scipy.ndimage import (binary_dilation, binary_erosion,
                            binary_closing, gaussian_filter,
                            label as nd_label)
from diffusers import AutoPipelineForImage2Image, AutoPipelineForInpainting
from transformers import (SegformerImageProcessor,
                          SegformerForSemanticSegmentation)
import gradio as gr

print(" Imports done!")
print(f" CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f" {p.name}  VRAM {p.total_memory/1e9:.1f} GB")

In [ ]:

def safe_open_image(src) -> Image.Image:
    """Open any source (path / Gradio file / PIL) → RGB, EXIF-corrected."""
    if isinstance(src, Image.Image):
        img = src
    elif hasattr(src, 'name'):
        img = Image.open(src.name)
    else:
        img = Image.open(str(src))
    img = ImageOps.exif_transpose(img)
    return img.convert('RGB')


def safe_to_pil(arr: np.ndarray) -> Image.Image:
    """Any numpy array → uint8 RGB PIL — handles float [0,1] and [0,255]."""
    arr = np.asarray(arr)
    if arr.dtype in (np.float16, np.float32, np.float64):
        arr = (arr * 255.0) if arr.max() <= 1.01 else arr
        arr = arr.clip(0, 255).astype(np.uint8)
    elif arr.dtype != np.uint8:
        arr = arr.astype(np.uint8)
    if arr.ndim == 3 and arr.shape[2] == 1:
        arr = arr[:, :, 0]
    return Image.fromarray(arr)


def pil_to_base64(img: Image.Image, fmt="JPEG", quality=88) -> str:
    """Encode PIL image → base64 data URI (for HTML embedding)."""
    buf  = io.BytesIO()
    img.save(buf, format=fmt, quality=quality)
    b64  = base64.b64encode(buf.getvalue()).decode()
    mime = "image/jpeg" if fmt.upper() == "JPEG" else "image/png"
    return f"data:{mime};base64,{b64}"


def save_png(img: Image.Image, path: str):
    """Always save as PNG so Drive files open as images, not binary files."""
    if not path.endswith('.png'):
        path = path.rsplit('.', 1)[0] + '.png'
    img.save(path, format='PNG')
    return path

In [ ]:
print(" Loading SDXL-Turbo …")
MODEL_ID = "stabilityai/sdxl-turbo"

pipe_i2i = AutoPipelineForImage2Image.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, variant="fp16",
).to("cuda")
pipe_i2i.enable_attention_slicing(1)
# VAE slicing reduces peak VRAM during decode
pipe_i2i.vae.enable_slicing()
print(" img2img ready!")

pipe_inp = AutoPipelineForInpainting.from_pipe(pipe_i2i).to("cuda")
pipe_inp.enable_attention_slicing(1)
pipe_inp.vae.enable_slicing()
print(" Inpainting ready!")

In [ ]:
SEG_ID = "nvidia/segformer-b5-finetuned-ade-640-640"
print(f" Loading SegFormer …")
seg_processor = SegformerImageProcessor.from_pretrained(SEG_ID)
seg_model     = SegformerForSemanticSegmentation.from_pretrained(SEG_ID)
seg_model     = seg_model.to("cuda").eval()
print(" SegFormer loaded!")


ADE20K_CLASSES = {
    "wall":         [0, 1],
    "ceiling":      [5, 2],
    "floor":        [3, 28, 13],
    "tile":         [3],
    "door":         [14],
    "window":       [8],
    "bed":          [7],
    "wardrobe":     [35],
    "cabinet":      [24, 62],
    "sofa":         [23],
    "chair":        [19],
    "table":        [15],
    "desk":         [33],
    "shelf":        [62],
    "lamp":         [36],
    "lighting":     [36, 82],
    "curtain":      [18],
    "rug":          [28],
    "pillow":       [57],
    "mirror":       [27],
    "sink":         [49],
    "bathtub":      [38],
    "toilet":       [65],
    "plant":        [17, 66],
    "stairs":       [53],
    "tv":           [89],
    "refrigerator": [50],
    "stove":        [116],
}

ADE20K_ALIASES = {
    "walls":"wall","ceilings":"ceiling","floors":"floor",
    "tiles":"tile","tiling":"tile","doors":"door","windows":"window",
    "beds":"bed","mattress":"bed","bedding":"bed",
    "wardrobes":"wardrobe","closet":"wardrobe","cupboard":"wardrobe",
    "cabinets":"cabinet","couch":"sofa","settee":"sofa","sofas":"sofa",
    "chairs":"chair","armchair":"chair","seating":"chair",
    "tables":"table","counter":"table","countertop":"table","desks":"desk",
    "shelves":"shelf","shelving":"shelf","bookcase":"shelf",
    "lamps":"lamp","chandelier":"lighting","light":"lighting",
    "lights":"lighting","pendant":"lighting",
    "curtains":"curtain","blinds":"curtain","drapes":"curtain",
    "rugs":"rug","carpet":"rug","pillows":"pillow",
    "cushion":"pillow","cushions":"pillow","mirrors":"mirror",
    "plants":"plant","flowers":"plant","tree":"plant",
    "television":"tv","screen":"tv","fridge":"refrigerator",
}

print(f" ADE20K: {len(ADE20K_CLASSES)} classes, {len(ADE20K_ALIASES)} aliases")

In [ ]:
SIDES = {
    "🔵 Front":  {"yaw":   0, "pitch":  0,  "fov": 90, "label": "Front View"},
    "🟢 Right":  {"yaw":  90, "pitch":  0,  "fov": 90, "label": "Right Wall"},
    "🔴 Back":   {"yaw": 180, "pitch":  0,  "fov": 90, "label": "Back Wall"},
    "🟡 Left":   {"yaw": -90, "pitch":  0,  "fov": 90, "label": "Left Wall"},
    "⚪ Top":    {"yaw":   0, "pitch": 60,  "fov": 90, "label": "Ceiling View"},
    "🟤 Bottom": {"yaw":   0, "pitch": -60, "fov": 90, "label": "Floor View"},
}
SIDE_NAMES = list(SIDES.keys())

SIDE_PROMPT_SUFFIX = {
    "🔵 Front":  "front-facing wall view, directly ahead perspective, ",
    "🟢 Right":  "right side wall view, camera facing right, ",
    "🔴 Back":   "back wall view, rear perspective, ",
    "🟡 Left":   "left side wall view, camera facing left, ",
    "⚪ Top":    "ceiling overhead view, looking up, ",
    "🟤 Bottom": "floor downward view, looking down, ",
}

In [ ]:
QUALITY_PREFIX = (
    "photorealistic interior design photograph, "
    "professional architectural photography, "
    "8k ultra-detailed, sharp focus, "
    "correct perspective, physically accurate lighting, "
    "high dynamic range, realistic materials and textures, "
)


PANO_PROMPT_BOOSTER = (
    "cinematic interior, award-winning architecture photo, "
    "vivid colors, high contrast, professionally lit, "
)

PANORAMIC_SUFFIX = (
    "equirectangular 360 panoramic interior, "
    "seamless edges, consistent lighting throughout, "
)
STYLE_EXPANSIONS = {
    "minimalist":   "clean lines, negative space, monochromatic palette, ",
    "scandinavian": "hygge atmosphere, light birch wood, soft neutrals, cozy textiles, ",
    "industrial":   "exposed brick, raw concrete, black steel, Edison bulbs, ",
    "luxury":       "opulent materials, marble surfaces, gold fixtures, dramatic lighting, ",
    "japanese":     "wabi-sabi aesthetic, natural wood, shoji screens, tatami, ",
    "coastal":      "whitewash wood, rattan, linen textiles, sea-glass tones, ",
    "bohemian":     "layered textiles, earthy palette, macrame, indoor plants, ",
    "mid-century":  "organic curves, walnut veneer, mustard accents, tapered legs, ",
    "art deco":     "geometric patterns, black and gold, lacquered surfaces, symmetry, ",
    "farmhouse":    "shiplap walls, reclaimed wood, galvanized metal, linen, ",
    "modern":       "sleek surfaces, neutral palette, statement lighting, open-plan, ",
}
BASE_NEG = (
    "blurry, out of focus, low quality, jpeg artefacts, compression noise, "
    "overexposed, underexposed, blown highlights, deep shadows, "
    "distorted perspective, fisheye, warped walls, "
    "cartoon, anime, painting, illustration, sketch, "
    "floating objects, impossible geometry, "
    "watermark, text, logo, signature, border, frame, "
    "people, person, human, face, hands, "
    "duplicate, tiling artefact, bad anatomy, ugly, deformed"
)
PANO_NEG_EXTRA = (
    ", visible seam, hard seam line, mismatched lighting across seam, "
    "abrupt colour change at border"
)


def enhance_prompt(user_prompt: str, mode: str = "standard",
                   side: str = None) -> tuple:
    lower  = user_prompt.lower()
    boost  = "".join(v for k, v in STYLE_EXPANSIONS.items() if k in lower)
    side_v = SIDE_PROMPT_SUFFIX.get(side, "") if side else ""
    pano_b = PANO_PROMPT_BOOSTER if mode == "panoramic" else ""
    suffix = PANORAMIC_SUFFIX if mode == "panoramic" else ""
    pos    = (QUALITY_PREFIX + pano_b + side_v + boost +
              user_prompt.strip() + ", " + suffix).strip(", ")
    neg    = BASE_NEG + (PANO_NEG_EXTRA if mode == "panoramic" else "")
    return pos, neg

In [ ]:

def load_panorama(src, max_width: int = 2048) -> Image.Image:
    img  = safe_open_image(src)
    w, h = img.size
    if abs(h - w // 2) > max(10, h * 0.05):
        img = img.resize((w, w // 2), Image.LANCZOS)
    if w > max_width:
        img = img.resize((max_width, max_width // 2), Image.LANCZOS)
    return img


def extract_perspective(pano: Image.Image,
                        yaw: float = 0, pitch: float = 0,
                        fov: float = 90, out_size: int = 768) -> Image.Image:
    arr  = np.array(pano, dtype=np.float32) / 255.0
    view = py360convert.e2p(
        arr, fov_deg=(fov, fov), u_deg=float(yaw), v_deg=float(pitch),
        out_hw=(out_size, out_size), in_rot_deg=0, mode='bilinear',
    )
    return safe_to_pil(view)


def perspective_to_equirectangular(pano: Image.Image,
                                   view: Image.Image,
                                   yaw: float = 0, pitch: float = 0,
                                   fov: float = 90) -> Image.Image:
    """Bilinear reproject edited view → full equirectangular."""
    orig = np.array(pano, dtype=np.float32)
    H, W = orig.shape[:2]
    S    = max(view.width, view.height)
    edited = np.array(view.resize((S, S), Image.LANCZOS), dtype=np.float32)

    lon   = np.linspace(-np.pi, np.pi, W, endpoint=False)
    lat   = np.linspace(np.pi / 2, -np.pi / 2, H)
    lon_g, lat_g = np.meshgrid(lon, lat)

    yr = np.deg2rad(yaw); pr = np.deg2rad(pitch); fr = np.deg2rad(fov / 2)
    x  =  np.cos(lat_g) * np.sin(lon_g)
    y  =  np.sin(lat_g)
    z  =  np.cos(lat_g) * np.cos(lon_g)
    xr =  x * np.cos(-yr) + z * np.sin(-yr)
    zr = -x * np.sin(-yr) + z * np.cos(-yr)
    yr2=  y * np.cos(pr)  - zr * np.sin(pr)
    zr2=  y * np.sin(pr)  + zr * np.cos(pr)

    in_front = zr2 > 0
    half     = S / 2.0
    px = ( xr  / (zr2 + 1e-8) / np.tan(fr) * 0.5 + 0.5) * (S - 1)
    py = (-yr2 / (zr2 + 1e-8) / np.tan(fr) * 0.5 + 0.5) * (S - 1)
    valid = in_front & (px >= 0) & (px < S - 1) & (py >= 0) & (py < S - 1)

    px_f = np.clip(px, 0, S - 1 - 1e-4)
    py_f = np.clip(py, 0, S - 1 - 1e-4)
    x0   = px_f.astype(np.int32);  x1 = (x0 + 1).clip(0, S - 1)
    y0   = py_f.astype(np.int32);  y1 = (y0 + 1).clip(0, S - 1)
    wx   = (px_f - x0)[:, :, np.newaxis]
    wy   = (py_f - y0)[:, :, np.newaxis]

    reprojected = np.zeros_like(orig)
    reprojected[valid] = (
        (1 - wy[valid]) * ((1 - wx[valid]) * edited[y0[valid], x0[valid]]
                           +     wx[valid]  * edited[y0[valid], x1[valid]])
        +     wy[valid]  * ((1 - wx[valid]) * edited[y1[valid], x0[valid]]
                            +    wx[valid]  * edited[y1[valid], x1[valid]])
    )

    cx    = (px - (half - 0.5)) / (half - 0.5)
    cy    = (py - (half - 0.5)) / (half - 0.5)
    w_map = np.exp(-2.0 * (cx ** 2 + cy ** 2))
    w_map[~valid] = 0.0
    w3    = w_map[:, :, np.newaxis]

    blended = orig * (1.0 - w3) + reprojected * w3
    return safe_to_pil(blended)


def multi_view_stitch(pano: Image.Image,
                      edit_views: list,
                      yaw_list: list,
                      pitch: float = 0,
                      fov: float = 90) -> Image.Image:
    """
    FIXED stitch: collect all edited views simultaneously with additive
    weight accumulation. Uses steeper Gaussian weights (exp(-5 * r²))
    so the center of each view gets near-100% replacement and seam
    blending is tight. Covered pixels are FULLY replaced by the edited
    colour — the original is only kept where no view reaches.
    """
    orig     = np.array(pano, dtype=np.float32)
    H, W     = orig.shape[:2]
    accum    = np.zeros_like(orig)
    w_accum  = np.zeros((H, W, 1), dtype=np.float32)

    for yaw, view in zip(yaw_list, edit_views):
        S    = max(view.width, view.height)
        edited = np.array(view.resize((S, S), Image.LANCZOS), dtype=np.float32)

        lon   = np.linspace(-np.pi, np.pi, W, endpoint=False)
        lat   = np.linspace(np.pi / 2, -np.pi / 2, H)
        lon_g, lat_g = np.meshgrid(lon, lat)

        yr = np.deg2rad(yaw); pr = np.deg2rad(pitch); fr = np.deg2rad(fov / 2)
        x  =  np.cos(lat_g) * np.sin(lon_g)
        y  =  np.sin(lat_g)
        z  =  np.cos(lat_g) * np.cos(lon_g)
        xr =  x * np.cos(-yr) + z * np.sin(-yr)
        zr = -x * np.sin(-yr) + z * np.cos(-yr)
        yr2=  y * np.cos(pr)  - zr * np.sin(pr)
        zr2=  y * np.sin(pr)  + zr * np.cos(pr)

        in_front = zr2 > 0
        half     = S / 2.0
        px = ( xr  / (zr2 + 1e-8) / np.tan(fr) * 0.5 + 0.5) * (S - 1)
        py = (-yr2 / (zr2 + 1e-8) / np.tan(fr) * 0.5 + 0.5) * (S - 1)
        valid = in_front & (px >= 0) & (px < S - 1) & (py >= 0) & (py < S - 1)

        px_f = np.clip(px, 0, S - 1 - 1e-4)
        py_f = np.clip(py, 0, S - 1 - 1e-4)
        x0 = px_f.astype(np.int32); x1 = (x0 + 1).clip(0, S - 1)
        y0 = py_f.astype(np.int32); y1 = (y0 + 1).clip(0, S - 1)
        wx = (px_f - x0)[:, :, np.newaxis]
        wy = (py_f - y0)[:, :, np.newaxis]

        reproj = np.zeros_like(orig)
        reproj[valid] = (
            (1 - wy[valid]) * ((1 - wx[valid]) * edited[y0[valid], x0[valid]]
                               +     wx[valid]  * edited[y0[valid], x1[valid]])
            +     wy[valid]  * ((1 - wx[valid]) * edited[y1[valid], x0[valid]]
                                +    wx[valid]  * edited[y1[valid], x1[valid]])
        )


        cx    = (px - (half - 0.5)) / (half - 0.5)
        cy    = (py - (half - 0.5)) / (half - 0.5)
        w_map = np.exp(-5.0 * (cx ** 2 + cy ** 2))
        w_map[~valid] = 0.0
        w3    = w_map[:, :, np.newaxis]

        accum   += reproj * w3
        w_accum += w3


    covered  = (w_accum[:, :, 0] > 1e-4)
    result   = orig.copy()

    result[covered] = (accum / w_accum.clip(min=1e-6))[covered]
    return safe_to_pil(result)


def _flush():
    torch.cuda.empty_cache(); gc.collect()


In [ ]:

def _run_i2i(image, prompt, neg, strength, steps, cfg,
             seed=None, size: int = 768) -> Image.Image:
    gen   = torch.Generator("cuda").manual_seed(seed) if seed is not None else None
    img_r = image.resize((size, size), Image.LANCZOS)
    out   = pipe_i2i(
        prompt=prompt, negative_prompt=neg,
        image=img_r, strength=strength,
        num_inference_steps=steps, guidance_scale=cfg,
        generator=gen,
    ).images[0]
    return out.resize(image.size, Image.LANCZOS)


def _run_inpaint(image, mask, prompt, neg,
                 strength, steps, cfg, seed=None,
                 size: int = 768) -> Image.Image:
    """
    BUG 4C FIX: pass padding_mask_crop=32 so the pipeline inpaints
    only the bounding box of the mask + 32 px padding at full resolution.
    This produces sharper, more detailed edits in the masked region.
    """
    gen    = torch.Generator("cuda").manual_seed(seed) if seed is not None else None
    img_r  = image.resize((size, size), Image.LANCZOS)
    mask_r = mask.resize((size, size), Image.LANCZOS)
    try:
        out = pipe_inp(
            prompt=prompt, negative_prompt=neg,
            image=img_r, mask_image=mask_r,
            strength=strength, num_inference_steps=steps,
            guidance_scale=cfg, generator=gen,
            padding_mask_crop=32,
        ).images[0]
    except TypeError:

        out = pipe_inp(
            prompt=prompt, negative_prompt=neg,
            image=img_r, mask_image=mask_r,
            strength=strength, num_inference_steps=steps,
            guidance_scale=cfg, generator=gen,
        ).images[0]
    return out.resize(image.size, Image.LANCZOS)


def apply_sdxl_edit(image, prompt, neg_prompt,
                    strength=0.70, steps=4, cfg=0.0,
                    seed=None, mode="standard", side=None):
    epos, eneg = enhance_prompt(prompt, mode=mode, side=side)
    full_neg   = (neg_prompt.strip() + ", " + eneg) if neg_prompt.strip() else eneg
    result     = _run_i2i(image, epos, full_neg, strength, steps, cfg, seed)
    _flush()
    return result


def apply_sdxl_panoramic_edit(pano, prompt, neg_prompt,
                               yaw_list=None, pitch=0, fov=90,
                               strength=0.82,
                               steps=6,
                               cfg=0.0, seed=None):
    """
    FIXED panoramic edit:
    1. Extract views from the ORIGINAL pano (prevents compounding artifacts).
    2. Collect ALL edited views first, then stitch simultaneously.
    3. Apply one final light img2img pass on the full stitched pano so that
       seam zones and polar regions that no 90° view covers also pick up
       the new style — this makes the output look like a completely new,
       fresh panorama instead of an overlay.
    """
    if yaw_list is None:
        yaw_list = [0, 90, 180, -90]

    orig_views, edit_views = [], []

    for i, yaw in enumerate(yaw_list):
        print(f"  🔄 Editing view {i+1}/{len(yaw_list)}  yaw={yaw}° …")

        view   = extract_perspective(pano, yaw=yaw, pitch=pitch, fov=fov, out_size=768)
        edited = apply_sdxl_edit(view, prompt, neg_prompt,
                                  strength=strength, steps=steps,
                                  cfg=cfg, seed=seed, mode="panoramic")
        orig_views.append(view)
        edit_views.append(edited)
        print(f"  ✅ View done  GPU {torch.cuda.memory_allocated()/1e9:.2f} GB")


    for pitch_v, label in [(55, "Top"), (-55, "Bottom")]:
        print(f"  🔄 Editing {label} pitch={pitch_v}° …")
        view   = extract_perspective(pano, yaw=0, pitch=pitch_v, fov=90, out_size=768)
        edited = apply_sdxl_edit(view, prompt, neg_prompt,
                                  strength=strength, steps=steps,
                                  cfg=cfg, seed=seed, mode="panoramic")
        yaw_list = list(yaw_list) + [0]
        edit_views.append(edited)
        print(f"  ✅ {label} view done")

    print("  🔀 Stitching all views simultaneously …")
    h_yaws = yaw_list[:4]
    h_edit = edit_views[:4]
    updated = multi_view_stitch(pano, h_edit, h_yaws, pitch=pitch, fov=fov)


    if len(edit_views) > 4:
        updated = multi_view_stitch(updated, [edit_views[4]], [0], pitch=55,  fov=90)
    if len(edit_views) > 5:
        updated = multi_view_stitch(updated, [edit_views[5]], [0], pitch=-55, fov=90)


    print("  🎨 Final full-pano harmonisation pass …")
    epos, eneg = enhance_prompt(prompt, mode="panoramic")
    full_neg   = (neg_prompt.strip() + ", " + eneg) if neg_prompt.strip() else eneg
    gen        = torch.Generator("cuda").manual_seed(seed) if seed is not None else None

    W, H       = updated.size
    tW, tH     = (min(W, 2048) // 64) * 64, (min(H, 1024) // 64) * 64
    pano_r     = updated.resize((tW, tH), Image.LANCZOS)
    harmonised = pipe_i2i(
        prompt=epos, negative_prompt=full_neg,
        image=pano_r, strength=0.28,
        num_inference_steps=max(steps, 6), guidance_scale=cfg,
        generator=gen,
    ).images[0].resize((W, H), Image.LANCZOS)
    _flush()

    return harmonised, orig_views, edit_views[:4]


def apply_reference_edit(image, prompt, neg_prompt, ref_img,
                          blend=0.30, strength=0.72, steps=4, cfg=0.0, seed=None):
    ref_r = ref_img.resize(image.size, Image.LANCZOS)
    mixed = Image.blend(image, ref_r, alpha=blend)
    mixed = mixed.filter(ImageFilter.UnsharpMask(radius=1, percent=120, threshold=2))
    return apply_sdxl_edit(mixed,
                           f"{prompt.strip()}, matching the style and palette of the reference image",
                           neg_prompt, strength=strength, steps=steps, cfg=cfg, seed=seed)


def make_comparison(images, labels, target_h=512) -> Image.Image:
    imgs = [img.resize((int(img.width * target_h / img.height), target_h), Image.LANCZOS)
            for img in images]
    SEP     = 4
    total_w = sum(i.width for i in imgs) + SEP * (len(imgs) - 1)
    canvas  = Image.new('RGB', (total_w, target_h + 48), (15, 15, 15))
    draw    = ImageDraw.Draw(canvas)
    x = 0
    for img, lbl in zip(imgs, labels):
        canvas.paste(img, (x, 48))
        draw.text((x + 9, 13), lbl, fill=(0, 0, 0))
        draw.text((x + 8, 12), lbl, fill=(240, 240, 240))
        x += img.width
        if x < total_w:
            draw.rectangle([x, 0, x + SEP - 1, target_h + 48], fill=(50, 50, 50))
            x += SEP
    return canvas


print(" SDXL wrappers ready!")

In [ ]:

def segment_image_tiled(image: Image.Image) -> np.ndarray:
    """
    Tiled SegFormer at native 640 px resolution.
    Single-pass for ≤ 768 px; tiled for larger images.
    """
    W, H = image.size
    if W <= 768 and H <= 768:
        seg_in  = image.resize((640, 640), Image.LANCZOS)
        inputs  = seg_processor(images=seg_in, return_tensors="pt").to("cuda")
        with torch.no_grad():
            logits = seg_model(**inputs).logits
        upsampled = F.interpolate(logits, size=(H, W),
                                   mode="bilinear", align_corners=False)
        return upsampled.argmax(1).squeeze(0).cpu().numpy().astype(np.int32)

    TILE = 640; STRIDE = 480; N_CLS = 150
    accum  = torch.zeros(1, N_CLS, H, W, device="cuda")
    counts = torch.zeros(1, 1,     H, W, device="cuda")

    xs = list(range(0, W - TILE, STRIDE)) + [max(0, W - TILE)]
    ys = list(range(0, H - TILE, STRIDE)) + [max(0, H - TILE)]

    for y in ys:
        for x in xs:
            tile_pil = image.crop((x, y, x + TILE, y + TILE))
            inp      = seg_processor(images=tile_pil, return_tensors="pt").to("cuda")
            with torch.no_grad():
                log_tile = seg_model(**inp).logits
            log_up = F.interpolate(log_tile, size=(TILE, TILE),
                                    mode="bilinear", align_corners=False)
            y2 = min(y + TILE, H); x2 = min(x + TILE, W)
            accum[:, :,  y:y2, x:x2] += log_up[:, :, :y2-y, :x2-x]
            counts[:, :, y:y2, x:x2] += 1.0

    averaged = accum / counts.clamp(min=1)
    return averaged.argmax(1).squeeze(0).cpu().numpy().astype(np.int32)


def resolve_target(text: str):
    t    = text.lower()
    keys = sorted(list(ADE20K_CLASSES) + list(ADE20K_ALIASES),
                  key=len, reverse=True)
    for k in keys:
        if k in t:
            canon = ADE20K_ALIASES.get(k, k)
            return canon, ADE20K_CLASSES[canon]
    raise ValueError(
        f"'{text}' not recognised. "
        f"Supported: {', '.join(sorted(ADE20K_CLASSES))}"
    )



_ELEMENT_CONFIG = {

    "wall":         (4,  11, False),
    "ceiling":      (4,  11, False),
    "floor":        (4,  11, False),
    "tile":         (4,   9, False),
    "door":         (6,   5, True),
    "window":       (6,   5, True),
    "bed":          (6,   7, False),
    "wardrobe":     (5,   7, True),
    "cabinet":      (5,   7, True),
    "sofa":         (6,   7, False),
    "chair":        (6,   5, True),
    "table":        (6,   5, True),
    "desk":         (6,   5, True),
    "shelf":        (5,   5, True),
    "lamp":         (7,   3, True),
    "lighting":     (7,   3, True),
    "curtain":      (5,   7, False),
    "rug":          (4,   9, False),
    "pillow":       (7,   3, False),
    "mirror":       (6,   5, True),
    "sink":         (6,   5, True),
    "bathtub":      (5,   7, False),
    "toilet":       (6,   5, True),
    "plant":        (7,   5, False),
    "stairs":       (4,   9, False),
    "tv":           (6,   5, True),
    "refrigerator": (5,   7, True),
    "stove":        (5,   7, True),
}


def _sigmoid_mask(arr: np.ndarray, sharpness: float = 12.0) -> np.ndarray:
    """Sigmoid-based boundary — sharper edge than gaussian for hard objects."""
    return 1.0 / (1.0 + np.exp(-sharpness * (arr - 0.5)))


def build_inpaint_mask(seg_map: np.ndarray,
                       class_ids: list,
                       canonical: str = "wall") -> Image.Image:
    """
    BUG 4b/4d FIX: element-adaptive closing kernel, adaptive edge sharpness.
    """
    H, W   = seg_map.shape
    binary = np.zeros((H, W), dtype=bool)
    for cid in class_ids:
        binary |= (seg_map == cid)

    coverage = binary.mean()
    print(f"  🗺️  Raw coverage: {coverage*100:.1f}%")
    if coverage < 0.001:
        print("    Very low coverage — element may not be visible")

    cfg = _ELEMENT_CONFIG.get(canonical, (6, 7, False))
    dil_r, close_k, edge_hard = cfg


    struct_close = np.ones((close_k, close_k), dtype=bool)
    struct3      = np.ones((3, 3), dtype=bool)

    closed  = binary_closing(binary, structure=struct_close, iterations=3)
    eroded  = binary_erosion(closed, structure=struct3, iterations=1)
    eroded  = np.where(eroded, closed, binary)
    dilated = binary_dilation(eroded, iterations=dil_r)

    if edge_hard:

        sigma     = max(1.5, dil_r * 0.5)
        feathered = gaussian_filter(dilated.astype(np.float32), sigma=sigma)
        mask_f    = _sigmoid_mask(feathered, sharpness=14.0)
    else:
        sigma     = max(2.0, dil_r * 0.8)
        feathered = gaussian_filter(dilated.astype(np.float32), sigma=sigma)
        mask_f    = (feathered >= 0.48).astype(np.float32)
        mask_f    = gaussian_filter(mask_f, sigma=1.5)

    mask_u8 = (mask_f * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(mask_u8, mode='L')


def composite_inpaint_result(original, inpainted, mask) -> Image.Image:
    orig_a = np.array(original, dtype=np.float32)
    inp_a  = np.array(inpainted.resize(original.size, Image.LANCZOS), dtype=np.float32)
    msk_a  = np.array(mask.resize(original.size, Image.LANCZOS), dtype=np.float32) / 255.0
    alpha  = msk_a[:, :, np.newaxis]
    return safe_to_pil(orig_a * (1 - alpha) + inp_a * alpha)


def overlay_mask_preview(image, mask,
                          color=(255, 60, 60), alpha=0.45) -> Image.Image:
    base   = image.convert("RGBA")
    msk_np = np.array(mask.resize(image.size, Image.LANCZOS))
    ov     = np.zeros((*msk_np.shape, 4), dtype=np.uint8)
    ov[msk_np > 32] = (*color, int(255 * alpha))
    return Image.alpha_composite(base, Image.fromarray(ov, "RGBA")).convert("RGB")


print(" Segmentation engine ready!")

In [ ]:


ELEMENT_PREFIXES = {
    "wall":         "interior room wall surface, ",
    "ceiling":      "interior room ceiling, overhead surface, ",
    "floor":        "interior floor surface, ",
    "tile":         "floor or wall tile surface, ",
    "door":         "interior door with frame, ",
    "window":       "interior window with frame, ",
    "bed":          "bedroom bed, frame and bedding, ",
    "wardrobe":     "bedroom wardrobe, built-in closet, ",
    "cabinet":      "storage cabinet or cupboard, ",
    "sofa":         "living room sofa, upholstered, ",
    "chair":        "chair, upholstered seat, ",
    "table":        "interior table, surface and legs, ",
    "desk":         "work desk furniture, ",
    "shelf":        "wall shelf, bookcase, ",
    "lamp":         "interior lamp, lighting fixture, ",
    "lighting":     "interior ceiling light, pendant, chandelier, ",
    "curtain":      "window curtain, drape or blind, ",
    "rug":          "floor rug, carpet, ",
    "pillow":       "decorative throw pillow, ",
    "mirror":       "wall mirror, framed, ",
    "sink":         "bathroom or kitchen sink, ",
    "bathtub":      "bathtub, freestanding or built-in, ",
    "toilet":       "toilet, porcelain, ",
    "plant":        "indoor plant, potted, ",
    "stairs":       "interior staircase, ",
    "tv":           "flat-screen television, ",
    "refrigerator": "kitchen refrigerator, ",
    "stove":        "kitchen stove, cooktop, ",
}


def apply_targeted_inpaint(image, target, prompt, neg,
                            strength=0.95, steps=12, cfg=8.0,
                            seed=None, side=None):
    canonical, class_ids = resolve_target(target)
    print(f"  🔍 '{canonical}'  IDs {class_ids}")

    seg_map  = segment_image_tiled(image)
    mask     = build_inpaint_mask(seg_map, class_ids, canonical)
    coverage = (np.array(mask) > 32).mean()
    print(f"  ✅ Mask coverage {coverage*100:.1f}%")

    if coverage < 0.002:
        raise ValueError(
            f"'{canonical}' not detected (coverage {coverage*100:.2f}%). "
            "Try a different view or element name."
        )

    prefix   = ELEMENT_PREFIXES.get(canonical, f"{canonical}, ")
    epos, eneg = enhance_prompt(prefix + prompt, side=side)
    full_neg = (neg.strip() + ", " + eneg) if neg.strip() else eneg

    print(f"  🎨 Inpainting … strength={strength} steps={steps} cfg={cfg}")
    inpainted = _run_inpaint(image, mask, epos, full_neg,
                              strength, steps, cfg, seed, size=768)
    final   = composite_inpaint_result(image, inpainted, mask)
    preview = overlay_mask_preview(image, mask)
    _flush()
    return final, mask, preview, canonical, float(coverage)


def apply_targeted_panoramic_inpaint(pano, target, prompt, neg,
                                      yaw_list=None, strength=0.95,
                                      steps=12, cfg=8.0, seed=None):
    if yaw_list is None:
        yaw_list = [0, 90, 180, -90]
    updated = pano.copy(); preview = None
    for i, yaw in enumerate(yaw_list):
        print(f"\n  🌐 Targeted pano view {i+1}/{len(yaw_list)} yaw={yaw}°")
        view = extract_perspective(updated, yaw=yaw, pitch=0, fov=90, out_size=768)
        try:
            final_v, mask, pv, canonical, cov = apply_targeted_inpaint(
                view, target, prompt, neg,
                strength=strength, steps=steps, cfg=cfg, seed=seed,
            )
            updated = perspective_to_equirectangular(updated, final_v,
                                                      yaw=yaw, pitch=0, fov=90)
            if i == 0: preview = pv
        except ValueError as e:
            print(f"  ⏭️  Skip yaw={yaw}: {e}")
        _flush()
    return updated, preview


print(" Targeted inpainting ready!")

In [ ]:

_state = {}


def _ts():    return datetime.now().strftime("%Y%m%d_%H%M%S")
def _seed(s): return int(s) if str(s).strip() else None


# ── TAB 1: SIDE VIEW EDIT ────────────────────────────────

def preview_side(pano_file, side_name):
    if pano_file is None:
        return None, "❌ Upload a panorama first."
    try:
        pano = load_panorama(pano_file)
        cfg  = SIDES[side_name]
        view = extract_perspective(pano, yaw=cfg["yaw"],
                                   pitch=cfg["pitch"], fov=cfg["fov"],
                                   out_size=768)
        return view, f"📸 {cfg['label']} (yaw={cfg['yaw']}° pitch={cfg['pitch']}°)"
    except Exception as e:
        return None, f"❌ {e}"


def run_side_edit(pano_file, side_name, prompt, neg,
                  strength, steps, cfg, seed_txt, stitch_back):
    if pano_file is None:
        return None, None, None, None, "❌ Upload a panorama first."
    try:
        ts   = _ts()
        pano = load_panorama(pano_file)
        side = SIDES[side_name]
        view = extract_perspective(pano, yaw=side["yaw"],
                                   pitch=side["pitch"], fov=side["fov"],
                                   out_size=768)
        print(f"🎨 Side edit | {side_name} | '{prompt[:60]}'")
        edited = apply_sdxl_edit(view, prompt, neg,
                                  strength=strength, steps=int(steps),
                                  cfg=cfg, seed=_seed(seed_txt),
                                  side=side_name)
        comp = make_comparison([view, edited], [f"📸 {side['label']}", "✨ Edited"])

        stitched_pano = None
        if stitch_back:
            stitched_pano = perspective_to_equirectangular(
                pano, edited, yaw=side["yaw"],
                pitch=side["pitch"], fov=side["fov"],
            )
            _state["last_pano"] = stitched_pano
            # BUG 3 FIX: save as PNG
            save_png(stitched_pano, f"{PROJECT_DIR}/outputs/side_pano_{ts}.png")

        _state["last_edited_view"] = edited
        save_png(view,   f"{PROJECT_DIR}/outputs/side_orig_{ts}.png")
        save_png(edited, f"{PROJECT_DIR}/outputs/side_edit_{ts}.png")
        save_png(comp,   f"{PROJECT_DIR}/outputs/side_comp_{ts}.png")

        status = (f"✅ {side['label']} edited!\n"
                  + ("   Stitched into full panorama.\n" if stitch_back else "")
                  + "💾 Saved as PNG.")
        return view, edited, comp, stitched_pano, status
    except Exception as e:
        return None, None, None, None, f"❌ {e}\n{traceback.format_exc()}"



# ── TAB 3: REFERENCE EDIT ────────────────────────────────

def run_reference_edit(pano_file, ref_file, prompt, neg,
                        strength, blend, steps, cfg, seed_txt,
                        side_name="🔵 Front"):
    if pano_file is None or ref_file is None:
        return None, None, None, "❌ Upload panorama AND reference image."
    try:
        ts    = _ts()
        pano  = load_panorama(pano_file)
        ref   = safe_open_image(ref_file)
        side  = SIDES.get(side_name, SIDES["🔵 Front"])
        view  = extract_perspective(pano,
                                    yaw=side["yaw"], pitch=side["pitch"],
                                    fov=side["fov"], out_size=768)
        print(f"🖼️ Reference edit | {side_name} | '{prompt[:60]}'")
        edited = apply_reference_edit(view, prompt, neg, ref,
                                       blend=blend, strength=strength,
                                       steps=int(steps), cfg=cfg,
                                       seed=_seed(seed_txt))
        comp = make_comparison(
            [view, ref.resize((view.width, view.height), Image.LANCZOS), edited],
            [f"📸 {side['label']}", "🖼️ Reference", "✨ Result"],
        )
        for img, n in [(view, "ref_orig"), (edited, "ref_edit"), (comp, "ref_comp")]:
            save_png(img, f"{PROJECT_DIR}/outputs/{n}_{ts}.png")
        return view, edited, comp, f"✅ Reference edit done! ({side['label']}) Saved as PNG."
    except Exception as e:
        return None, None, None, f"❌ {e}\n{traceback.format_exc()}"


# ── TAB 4: TARGETED ELEMENT EDIT ─────────────────────────

def run_targeted_edit(pano_file, flat_file, target, prompt, neg,
                       strength, steps, cfg, seed_txt, panoramic_mode,
                       side_name="🔵 Front"):
    if pano_file is None and flat_file is None:
        return None, None, None, None, "❌ Upload a panorama or flat photo."
    if not target.strip():
        return None, None, None, None, "❌ Enter element (e.g. 'wall')."
    if not prompt.strip():
        return None, None, None, None, "❌ Describe how it should look."
    try:
        ts = _ts(); seed = _seed(seed_txt)

        if flat_file is not None:
            image = safe_open_image(flat_file).resize((768, 768), Image.LANCZOS)
            final, mask, preview, canonical, cov = apply_targeted_inpaint(
                image, target, prompt, neg,
                strength=strength, steps=int(steps), cfg=cfg, seed=seed,
            )
            comp = make_comparison([image, final],
                                    ["📸 Original", f"✨ {canonical.title()}"])
            slug = canonical.replace(" ", "_")
            for img, n in [(image, f"tgt_orig_{slug}"), (final, f"tgt_edit_{slug}"),
                           (comp,  f"tgt_comp_{slug}"), (preview, f"preview_{slug}")]:
                save_png(img, f"{PROJECT_DIR}/outputs/{n}_{ts}.png")
            return image, final, comp, preview, (
                f"✅ Done! Element:{canonical}  Coverage:{cov*100:.1f}%\n💾 PNG saved.")

        pano = load_panorama(pano_file)
        if panoramic_mode:
            updated, preview = apply_targeted_panoramic_inpaint(
                pano, target, prompt, neg,
                yaw_list=[0, 90, 180, -90],
                strength=strength, steps=int(steps), cfg=cfg, seed=seed,
            )
            _state["last_pano"] = updated
            comp = make_comparison([pano, updated],
                                    ["📸 Original 360°", f"✨ {target.title()} Edited"],
                                    target_h=400)
            slug = target.strip().replace(" ", "_")
            save_png(pano,    f"{PROJECT_DIR}/outputs/tgt_pano_orig_{slug}_{ts}.png")
            save_png(updated, f"{PROJECT_DIR}/outputs/tgt_pano_edit_{slug}_{ts}.png")
            save_png(comp,    f"{PROJECT_DIR}/outputs/tgt_pano_comp_{slug}_{ts}.png")
            return pano, updated, comp, preview, "✅ Panoramic targeted edit done!\n💾 PNG saved."
        else:
            side = SIDES.get(side_name, SIDES["🔵 Front"])
            print(f"🎯 Targeted edit | {side_name} (yaw={side['yaw']}°) | '{target}' | '{prompt[:50]}'")
            view = extract_perspective(pano,
                                       yaw=side["yaw"], pitch=side["pitch"],
                                       fov=side["fov"], out_size=768)
            final, mask, preview, canonical, cov = apply_targeted_inpaint(
                view, target, prompt, neg,
                strength=strength, steps=int(steps), cfg=cfg, seed=seed,
                side=side_name,
            )
            comp = make_comparison([view, final],
                                    [f"📸 {side['label']}", f"✨ {canonical.title()}"])
            slug = canonical.replace(" ", "_")
            for img, n in [(view, f"tgt_orig_{slug}"), (final, f"tgt_edit_{slug}"),
                           (comp,  f"tgt_comp_{slug}"), (preview, f"preview_{slug}")]:
                save_png(img, f"{PROJECT_DIR}/outputs/{n}_{ts}.png")
            return view, final, comp, preview, (
                f"✅ Done! Element:{canonical}  Side:{side['label']}  Coverage:{cov*100:.1f}%\n💾 PNG saved.")
    except ValueError as ve:
        return None, None, None, None, f"❌ {ve}"
    except Exception as e:
        return None, None, None, None, f"❌ {e}\n{traceback.format_exc()}"




In [ ]:

PROMPT_EXAMPLES = [
    "modern minimalist living room, warm oak floors, soft natural light, cream walls",
    "luxury penthouse, marble floors, gold accents, floor-to-ceiling windows",
    "scandinavian hygge bedroom, light birch wood, cream tones, cozy wool textiles",
    "industrial loft kitchen, exposed concrete, black steel shelving, Edison bulbs",
    "japanese zen room, tatami mats, shoji screens, bonsai, raked gravel",
    "coastal boho living room, whitewash wood, rattan furniture, sea-blue accents",
    "mid-century modern study, walnut furniture, mustard accents, parquet floor",
    "art deco bathroom, black and gold tile, freestanding tub, brass fixtures",
    "farmhouse kitchen, shiplap walls, reclaimed oak, open shelves, copper cookware",
    "contemporary bedroom, dark walls, built-in lighting, linen bedding",
]

DEFAULT_NEG = (
    "blurry, low quality, distorted, cartoon, watermark, text, people, "
    "overexposed, bad proportions, floating objects"
)

TARGETED_EXAMPLES = [
    ["wall",      "sage green matte painted walls, smooth plaster, soft ambient light"],
    ["wall",      "exposed red brick, industrial style, aged texture, moody lighting"],
    ["ceiling",   "white coffered ceiling, decorative moulding, recessed LED lighting"],
    ["ceiling",   "dark navy blue ceiling, gold cornice trim, dramatic atmosphere"],
    ["floor",     "warm herringbone oak parquet flooring, natural finish, glossy sheen"],
    ["floor",     "large format white marble tiles, grey veining, high gloss"],
    ["tile",      "hexagonal black and white mosaic tiles, classic bathroom style"],
    ["bed",       "king size upholstered bed, ivory linen headboard, neutral bedding"],
    ["wardrobe",  "floor to ceiling white built-in wardrobe, brushed gold handles"],
    ["sofa",      "L-shaped sectional, velvet emerald green, gold tapered legs"],
    ["chair",     "Eames lounge chair, dark walnut, black leather upholstery"],
    ["curtain",   "full-length ivory linen curtains, soft natural light diffusion"],
    ["rug",       "large Persian rug, deep burgundy and navy, ornate pattern"],
    ["lamp",      "modern brushed brass floor lamp, white drum shade, warm glow"],
    ["table",     "round white marble dining table, brushed gold legs"],
    ["door",      "solid dark oak door, brushed nickel handle, modern style"],
    ["window",    "tall arched window, white frame, sheer curtain, bright daylight"],
    ["plant",     "large fiddle leaf fig tree in white ceramic pot"],
    ["mirror",    "large round arch mirror, thin brass frame"],
    ["tv",        "large 85-inch OLED TV on floating walnut media console"],
]

TARGETED_TIPS = """
**🎯 How Targeted Edit works**
1. **SegFormer** (tiled 640 px native) segments your image into 150 semantic classes
2. Adaptive mask: element-specific closing kernel → speckle erosion → sigmoid/gaussian feather
3. **SDXL-Turbo** inpaints ONLY inside the mask using bounding-box crop for sharp detail
4. Float-alpha composite → zero bleed outside the mask

| Setting | Recommendation |
|---------|----------------|
| Target  | *wall · ceiling · floor · bed · wardrobe · sofa · rug · curtain · lamp · door · window · plant · mirror · tv* |
| Prompt  | Describe **only** that element |
| Strength| 0.90–1.0 |
| Steps   | 10–15 |
| Guidance| 7–9 |
"""

CSS = """
.tab-header   { font-size:1.05rem; font-weight:600; }
footer        { display:none; }
#pano-out img { max-width:100%; height:auto !important; }
#viewer-frame { border-radius:8px; overflow:hidden; background:#111; }
"""



with gr.Blocks(title="🏠 360° Room Editor v6",
               theme=gr.themes.Soft(), css=CSS) as demo:

    gr.Markdown("""
# 🏠 360° AI Room Editor v6
**Side View Selector · 3D Viewer (fixed) · Panoramic Edit (fixed) · Precision Masking**
SegFormer neural segmentation + SDXL-Turbo inpainting — PNG downloads
---
""")

    with gr.Tabs():

        # ════════════════════════════════════════════
        # TAB 1 — SIDE VIEW EDIT
        # ════════════════════════════════════════════
        with gr.TabItem("🧭 Side View Edit", elem_classes="tab-header"):
            gr.Markdown("""
Select **which side of the room** to edit.
Click **👁 Preview** first — the selected view appears instantly so you
know exactly what you're editing before running AI.
""")
            with gr.Row():
                with gr.Column(scale=1):
                    sv_pano  = gr.File(label="📤 360° Panorama",
                                       file_types=[".jpg", ".jpeg", ".png"])
                    gr.Markdown("**🧭 Choose Side**")
                    sv_side  = gr.Radio(choices=SIDE_NAMES, value="🔵 Front",
                                        label="Room Side to Edit",
                                        info="Which wall / surface to edit")
                    sv_prev_btn = gr.Button("👁 Preview Selected Side",
                                            variant="secondary", size="sm")
                    sv_prev_st  = gr.Textbox(label="Preview Info",
                                             max_lines=2, interactive=False)
                    gr.Markdown("**🎨 Edit Settings**")
                    sv_prompt = gr.Textbox(label="Prompt", lines=3,
                                           value=PROMPT_EXAMPLES[0])
                    sv_neg    = gr.Textbox(label="🚫 Negative", lines=2,
                                           value=DEFAULT_NEG)
                    with gr.Row():
                        sv_str = gr.Slider(0.30, 0.95, 0.68, step=0.01, label="Strength")
                        sv_stp = gr.Slider(1, 20, 4, step=1, label="Steps")
                    sv_cfg    = gr.Slider(0.0, 8.0, 0.0, step=0.5, label="Guidance")
                    sv_seed   = gr.Textbox(label="🎲 Seed", value="", max_lines=1)
                    sv_stitch = gr.Checkbox(
                        label="🌐 Stitch edit into full panorama",
                        value=True,
                        info="Reprojected back → equirectangular panorama updated",
                    )
                    sv_btn    = gr.Button("⚡ Edit This Side", variant="primary", size="lg")
                    sv_status = gr.Textbox(label="Status", lines=3, interactive=False)

                with gr.Column(scale=2):
                    gr.Markdown("**Preview & Result**")
                    with gr.Row():
                        sv_orig = gr.Image(label="📸 Selected Side (before)",
                                           type="pil", format="png", height=380)
                        sv_edit = gr.Image(label="✨ AI Edited Side",
                                           type="pil", format="png", height=380)
                    sv_comp     = gr.Image(label="📊 Before vs After",
                                           type="pil", format="png")
                    sv_pano_out = gr.Image(label="🌐 Full Panorama (stitched)",
                                           type="pil", format="png",
                                           elem_id="pano-out")

            gr.Markdown("""
> **Sides:** Front=0° · Right=90° · Back=180° · Left=-90° · Top=pitch+60° · Bottom=pitch-60°
> Use **Stitch Back ✅** then open Tab 5 to view changes in 3D.
""")
            gr.Examples([[e] for e in PROMPT_EXAMPLES],
                        inputs=[sv_prompt], label="💡 Prompt Examples")
            sv_prev_btn.click(preview_side, [sv_pano, sv_side],
                               [sv_orig, sv_prev_st])
            sv_btn.click(run_side_edit,
                         [sv_pano, sv_side, sv_prompt, sv_neg,
                          sv_str, sv_stp, sv_cfg, sv_seed, sv_stitch],
                         [sv_orig, sv_edit, sv_comp, sv_pano_out, sv_status])



        # ════════════════════════════════════════════
        # TAB 3 — REFERENCE EDIT
        # ════════════════════════════════════════════
        with gr.TabItem("🖼️ Reference Image Edit", elem_classes="tab-header"):
            with gr.Row():
                with gr.Column(scale=1):
                    rr_pano   = gr.File(label="📤 Your Panorama",
                                        file_types=[".jpg", ".jpeg", ".png"])
                    rr_ref    = gr.File(label="🖼️ Reference Style Photo",
                                        file_types=[".jpg", ".jpeg", ".png"])
                    gr.Markdown("**🧭 Choose Room Side to Edit**")
                    rr_side   = gr.Radio(choices=SIDE_NAMES, value="🔵 Front",
                                         label="Room Side",
                                         info="Which view of the room to apply the reference style to")
                    rr_prompt = gr.Textbox(label="🎨 Prompt", lines=3,
                                           value="luxury modern room matching the reference style")
                    rr_neg    = gr.Textbox(label="🚫 Negative", lines=2, value=DEFAULT_NEG)
                    rr_blend  = gr.Slider(0.10, 0.60, 0.30, step=0.05,
                                          label="Reference Blend")
                    with gr.Row():
                        rr_str = gr.Slider(0.30, 0.95, 0.72, step=0.01, label="Strength")
                        rr_stp = gr.Slider(1, 20, 4, step=1, label="Steps")
                    rr_cfg    = gr.Slider(0.0, 8.0, 0.0, step=0.5, label="Guidance")
                    rr_seed   = gr.Textbox(label="🎲 Seed", value="", max_lines=1)
                    rr_btn    = gr.Button("🖼️ Generate Reference Edit",
                                          variant="primary", size="lg")
                    rr_status = gr.Textbox(label="Status", lines=3, interactive=False)
                with gr.Column(scale=2):
                    with gr.Row():
                        rr_orig = gr.Image(label="📸 Selected Room Side",
                                           type="pil", format="png", height=380)
                        rr_edit = gr.Image(label="✨ Result",
                                           type="pil", format="png", height=380)
                    rr_comp = gr.Image(label="📊 Comparison",
                                        type="pil", format="png")
            gr.Markdown("""
> **Sides:** Front=0° · Right=90° · Back=180° · Left=-90° · Top=pitch+60° · Bottom=pitch-60°
> Pick the side whose style you want to match against the reference photo.
""")
            rr_btn.click(run_reference_edit,
                         [rr_pano, rr_ref, rr_prompt, rr_neg,
                          rr_str, rr_blend, rr_stp, rr_cfg, rr_seed, rr_side],
                         [rr_orig, rr_edit, rr_comp, rr_status])

        # ════════════════════════════════════════════
        # TAB 4 — TARGETED ELEMENT EDIT
        # ════════════════════════════════════════════
        with gr.TabItem("🎯 Targeted Element Edit", elem_classes="tab-header"):
            gr.Markdown("""
### Change ONE element — SegFormer neural segmentation locates it, SDXL-Turbo inpaints only that region.
Everything outside the detected region is **pixel-perfectly preserved** (float-alpha composite + bounding-box inpaint crop).
""")
            with gr.Row():
                with gr.Column(scale=1):
                    with gr.Group():
                        gr.Markdown("**📂 Image Source**")
                        te_pano = gr.File(label="📤 360° Panorama",
                                          file_types=[".jpg", ".jpeg", ".png"])
                        te_flat = gr.File(label="🖼️ Flat Room Photo",
                                          file_types=[".jpg", ".jpeg", ".png"])
                        te_mode = gr.Checkbox(
                            label="🌐 Panoramic mode (all 4 views + stitch)",
                            value=False,
                        )
                    with gr.Group():
                        gr.Markdown("**🧭 Room Side to Edit**")
                        te_side = gr.Radio(
                            choices=SIDE_NAMES, value="🔵 Front",
                            label="Room Side",
                            info="Active in single-view mode (panoramic mode edits all 4 sides automatically)",
                        )
                    with gr.Group():
                        gr.Markdown("**🎯 What to change**")
                        te_target = gr.Textbox(
                            label="Element", value="wall", max_lines=1,
                            placeholder="wall  ceiling  floor  bed  sofa  wardrobe  table …",
                        )
                        te_prompt = gr.Textbox(
                            label="✏️ How it should look", lines=3,
                            placeholder="Describe ONLY this element",
                        )
                        te_neg = gr.Textbox(label="🚫 Negative", lines=2, value=DEFAULT_NEG)
                    with gr.Group():
                        gr.Markdown("**⚙️ Settings**")
                        te_str  = gr.Slider(0.70, 1.00, 0.95, step=0.01,
                                            label="Strength (0.90–1.0)")
                        with gr.Row():
                            te_stp = gr.Slider(4, 30, 12, step=1,
                                               label="Steps (10–15)")
                            te_cfg = gr.Slider(1.0, 9.0, 8.0, step=0.5,
                                               label="Guidance (7–9)")
                        te_seed = gr.Textbox(label="🎲 Seed", value="", max_lines=1)
                    te_btn    = gr.Button("🎯 Apply Targeted Edit",
                                          variant="primary", size="lg")
                    te_status = gr.Textbox(label="📋 Status", lines=6, interactive=False)

                with gr.Column(scale=2):
                    with gr.Row():
                        te_orig   = gr.Image(label="📸 Original",
                                             type="pil", format="png", height=380)
                        te_result = gr.Image(label="✨ Edited",
                                             type="pil", format="png", height=380)
                    te_comp    = gr.Image(label="📊 Before vs After",
                                          type="pil", format="png")
                    te_preview = gr.Image(label="🔴 Mask Preview (red = edit zone)",
                                          type="pil", format="png")

            gr.Markdown(TARGETED_TIPS)
            gr.Examples(TARGETED_EXAMPLES, inputs=[te_target, te_prompt],
                        label="💡 Examples — click to load")
            te_btn.click(run_targeted_edit,
                         [te_pano, te_flat, te_target, te_prompt, te_neg,
                          te_str, te_stp, te_cfg, te_seed, te_mode, te_side],
                         [te_orig, te_result, te_comp, te_preview, te_status])



print("UI ready!")
demo.launch(share=True, debug=False)